# 01: Machine Learning Fundamentals

This notebook covers the basics of building ML models with scikit-learn:
- Linear regression for predicting continuous values
- Random forests for more complex patterns
- Logistic regression for classification
- Train/test splitting for proper evaluation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_squared_error, accuracy_score, classification_report
from sklearn.datasets import make_classification, make_regression

## Part 1: Linear Regression

Linear regression fits a straight line (or hyperplane) to data. It models the relationship:

$$y = w_1 x_1 + w_2 x_2 + \ldots + w_n x_n + b$$

The model learns the weights $w$ and bias $b$ that minimize prediction error.

In [ ]:
# Generate synthetic regression data: predict house size from features
np.random.seed(42)
n_samples = 200

# Features: square footage, number of bedrooms, age of house
sqft = np.random.uniform(500, 3500, n_samples)
bedrooms = np.random.randint(1, 6, n_samples)
age = np.random.uniform(0, 50, n_samples)

# True relationship: price depends on sqft, bedrooms, and age
# (in thousands of dollars)
price = 0.15 * sqft + 20 * bedrooms - 1.5 * age + np.random.normal(0, 30, n_samples)

# Combine features into a matrix
X_reg = np.column_stack([sqft, bedrooms, age])
y_reg = price

print(f"Feature matrix shape: {X_reg.shape}")
print(f"Target shape: {y_reg.shape}")
print(f"\nFirst 5 samples:")
print(f"  Sqft:    {sqft[:5]}")
print(f"  Beds:    {bedrooms[:5]}")
print(f"  Age:     {age[:5]}")
print(f"  Price:   {price[:5].round(1)}")

In [ ]:
# Split data into training and test sets (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples:     {X_test.shape[0]}")

In [ ]:
# Train a linear regression model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Examine learned coefficients
feature_names = ["sqft", "bedrooms", "age"]
print("Learned coefficients:")
for name, coef in zip(feature_names, lr_model.coef_):
    print(f"  {name:12s}: {coef:+.4f}")
print(f"  {'bias':12s}: {lr_model.intercept_:+.4f}")

print(f"\nTrue coefficients: sqft=0.15, bedrooms=20, age=-1.5")
print("The model learned coefficients close to the true values!")

In [ ]:
# Evaluate on test set
y_pred = lr_model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print(f"Test RMSE: ${rmse:.2f}k")
print(f"This means predictions are typically off by ~${rmse:.0f}k")

# Visualize predictions vs actual
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.6, edgecolors="k", linewidth=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
         "r--", linewidth=2, label="Perfect predictions")
plt.xlabel("Actual Price ($k)")
plt.ylabel("Predicted Price ($k)")
plt.title("Linear Regression: Predicted vs Actual")
plt.legend()
plt.tight_layout()
plt.show()

## Part 2: Random Forest Classifier

Random forests are an ensemble method that combines many decision trees. Each tree votes, and the majority wins. This reduces overfitting compared to a single decision tree.

In [ ]:
# Generate synthetic classification data
X_cls, y_cls = make_classification(
    n_samples=500,
    n_features=10,
    n_informative=5,
    n_redundant=2,
    n_classes=2,
    random_state=42,
    class_sep=1.5
)

print(f"Feature matrix shape: {X_cls.shape}")
print(f"Class distribution: {np.bincount(y_cls)}")
print(f"  Class 0 (negative): {np.sum(y_cls == 0)} samples")
print(f"  Class 1 (positive): {np.sum(y_cls == 1)} samples")

In [ ]:
# Split and train
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42
)

# Train a Random Forest with 100 trees
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)
rf_model.fit(X_train_c, y_train_c)

# Predict
y_pred_rf = rf_model.predict(X_test_c)

accuracy = accuracy_score(y_test_c, y_pred_rf)
print(f"Random Forest Accuracy: {accuracy:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test_c, y_pred_rf, target_names=["Negative", "Positive"]))

In [ ]:
# Feature importance: which features matter most?
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 5))
plt.bar(range(len(importances)), importances[indices], align="center")
plt.xticks(range(len(importances)), [f"Feature {i}" for i in indices])
plt.ylabel("Importance")
plt.title("Random Forest Feature Importances")
plt.tight_layout()
plt.show()

## Part 3: Logistic Regression (Classification)

Despite the name, logistic regression is a **classification** algorithm. It outputs probabilities using the sigmoid function:

$$P(y=1) = \frac{1}{1 + e^{-(w \cdot x + b)}}$$

The probability is thresholded (default 0.5) to produce a class label.

In [ ]:
# Generate simple 2D classification data for visualization
X_2d, y_2d = make_classification(
    n_samples=200,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    n_classes=2,
    random_state=42,
    class_sep=2.0
)

X_tr, X_te, y_tr, y_te = train_test_split(X_2d, y_2d, test_size=0.2, random_state=42)

# Train logistic regression
log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_tr, y_tr)

y_pred_lr = log_reg.predict(X_te)
y_prob_lr = log_reg.predict_proba(X_te)[:, 1]

print(f"Accuracy: {accuracy_score(y_te, y_pred_lr):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_te, y_pred_lr))

In [ ]:
# Visualize decision boundary
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Data points
colors = ["#3498db", "#e74c3c"]
for cls in [0, 1]:
    mask = y_te == cls
    axes[0].scatter(X_te[mask, 0], X_te[mask, 1],
                    c=colors[cls], label=f"Class {cls}",
                    edgecolors="k", linewidth=0.5, alpha=0.7)
axes[0].set_xlabel("Feature 0")
axes[0].set_ylabel("Feature 1")
axes[0].set_title("Test Data")
axes[0].legend()

# Plot 2: Predicted probabilities
scatter = axes[1].scatter(X_te[:, 0], X_te[:, 1],
                          c=y_prob_lr, cmap="RdBu_r", edgecolors="k",
                          linewidth=0.5, alpha=0.7)
plt.colorbar(scatter, ax=axes[1], label="P(Positive)")
axes[1].set_xlabel("Feature 0")
axes[1].set_ylabel("Feature 1")
axes[1].set_title("Predicted Probability of Class 1")

plt.tight_layout()
plt.show()

## Part 4: Comparing Models

Different algorithms have different strengths. Let's compare on the same dataset.

In [ ]:
# Compare models on the classification task
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM (RBF)": SVC(kernel="rbf", random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train_c, y_train_c)
    train_acc = model.score(X_train_c, y_train_c)
    test_acc = model.score(X_test_c, y_test_c)
    results[name] = {"train": train_acc, "test": test_acc}
    print(f"{name:25s} | Train: {train_acc:.4f} | Test: {test_acc:.4f}")

print("\nNote: If train accuracy >> test accuracy, the model is overfitting.")

In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(10, 5))
model_names = list(results.keys())
train_scores = [results[m]["train"] for m in model_names]
test_scores = [results[m]["test"] for m in model_names]

x = np.arange(len(model_names))
width = 0.35

bars1 = ax.bar(x - width/2, train_scores, width, label="Train", color="#3498db")
bars2 = ax.bar(x + width/2, test_scores, width, label="Test", color="#e74c3c")

ax.set_ylabel("Accuracy")
ax.set_title("Model Comparison: Train vs Test Accuracy")
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=15, ha="right")
ax.legend()
ax.set_ylim(0.5, 1.05)
plt.tight_layout()
plt.show()

## Key Takeaways

1. **Linear regression** learns a linear relationship between features and a continuous target.
2. **Logistic regression** learns a linear decision boundary for classification (outputs probabilities).
3. **Random forests** combine many decision trees for better generalization.
4. **Always split data** into train/test before evaluation — never evaluate on training data.
5. **Compare models** — no single algorithm is best for all problems.

Next: [02-model-evaluation.ipynb](02-model-evaluation.ipynb) — deep dive into evaluation metrics.